In [1]:
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer, AlbertModel, AlbertTokenizer, AlbertForSequenceClassification, Trainer, TrainingArguments
from evaluate import load
from transformers import AutoTokenizer

c:\Users\sin2x\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Проверка наличия GPU
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
# 1. Загрузка и подготовка данных
data = pd.read_csv('data/youtube.csv')
print(data.head())  # Проверка структуры данных

# Объединение заголовков и описаний в один текст
data['text'] = data['title'] + " " + data['description']

# Разделение данных на тренировочную и тестовую выборки
train_texts, test_texts, train_labels, test_labels = train_test_split(
    data['text'], data['category'], test_size=0.2, random_state=42
)

# Преобразование меток категорий в числовые значения
le = LabelEncoder()
train_labels_enc = le.fit_transform(train_labels)
test_labels_enc = le.transform(test_labels)

          link                                              title  \
0      JLZlCZ0  Ep 1| Travelling through North East India | Of...   
1  i9E_Blai8vk      Welcome to Bali | Travel Vlog | Priscilla Lee   
2   r284c-q8oY  My Solo Trip to ALASKA | Cruising From Vancouv...   
3   Qmi-Xwq-ME   Traveling to the Happiest Country in the World!!   
4  _lcOX55Ef70  Solo in Paro Bhutan | Tiger's Nest visit | Bhu...   

                                         description category  
0  Tanya Khanijow\n671K subscribers\nSUBSCRIBE\nT...   travel  
1  Priscilla Lee\n45.6K subscribers\nSUBSCRIBE\n*...   travel  
2  Allison Anderson\n588K subscribers\nSUBSCRIBE\...   travel  
3  Yes Theory\n6.65M subscribers\nSUBSCRIBE\n*BLA...   travel  
4  Tanya Khanijow\n671K subscribers\nSUBSCRIBE\nH...   travel  


In [4]:
# 2. Токенизация текста
tokenizer = AutoTokenizer.from_pretrained("albert-base-v2")
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128, return_tensors='pt')
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=128, return_tensors='pt')

In [5]:
# 3. Создание Dataset
class YouTubeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = YouTubeDataset(train_encodings, train_labels_enc)
test_dataset = YouTubeDataset(test_encodings, test_labels_enc)

In [6]:
# 4. Загрузка модели и установка параметров обучения
model = AlbertForSequenceClassification.from_pretrained(
    'albert-base-v2',
    num_labels=len(le.classes_)
).to(device)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy='epoch',
    logging_dir='./logs',
    logging_steps=10,
    report_to='none',
    save_strategy='epoch',
)

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
# 5. Функция вычисления метрик
def compute_metrics(eval_pred):
    metric_accuracy = load("accuracy")
    metric_precision = load("precision")
    metric_recall = load("recall")
    metric_f1 = load("f1")

    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)

    accuracy = metric_accuracy.compute(predictions=predictions, references=labels)
    precision = metric_precision.compute(predictions=predictions, references=labels, average='macro')
    recall = metric_recall.compute(predictions=predictions, references=labels, average='macro')
    f1 = metric_f1.compute(predictions=predictions, references=labels, average='macro')

    results = {
        "accuracy": accuracy["accuracy"],
        "precision": precision["precision"],
        "recall": recall["recall"],
        "f1": f1["f1"]
    }
    
    return results

In [8]:
# 6. Инициализация Trainer и обучение модели
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()  # Обучение модели

c:\Users\sin2x\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.331800,0.121813,0.970833,0.975290,0.970816,0.972987
2,0.312200,0.126422,0.973611,0.971969,0.976741,0.974117
3,0.102900,0.130357,0.979167,0.979248,0.981727,0.980367


c:\Users\sin2x\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\sin2x\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=1080, training_loss=0.2895839760588758, metrics={'train_runtime': 2417.4185, 'train_samples_per_second': 3.573, 'train_steps_per_second': 0.447, 'total_flos': 51612178111488.0, 'train_loss': 0.2895839760588758, 'epoch': 3.0})

In [11]:
# 7. Оценка модели и сохранение метрик
eval_results = trainer.evaluate()
metrics_df = pd.DataFrame([eval_results])
metrics_df.to_csv("training_metrics.csv", index=False)
print("Метрики сохранены в training_metrics.csv")

c:\Users\sin2x\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Метрики сохранены в training_metrics.csv


In [12]:
# 8. Сохранение модели и токенизатора
model.save_pretrained('./youtube-category-model')
tokenizer.save_pretrained('./youtube-category-model')
print("Модель и токенизатор сохранены.")

Модель и токенизатор сохранены.
